# 3.2 Document Pipeline - Build the RAG ingestion engine

## Project Role: DocumentPipeline feeds documents into HybridSearcher for the final project

In [ ]:
import sys, os
sys.path.insert(0, "..")
from config import get_client; client = get_client()
from agent_project import *
from agent_project.tools import create_default_registry
print(f"LLM: {client.name} | modules loaded")


In [ ]:
from agent_project.pipeline import DocumentPipeline
pipeline = DocumentPipeline(chunk_size=300, chunk_overlap=50)

KNOWLEDGE = [
  {"source":"ai_basics.md","content":"# AI Basics\nAI creates intelligent systems. ML trains models from data. DL uses multi-layer neural networks. CNNs process images, RNNs/Transformers handle sequences."},
  {"source":"llm_guide.md","content":"# LLM Guide\nLLMs have billions of parameters. Transformer by Vaswani et al. (2017). Self-Attention enables full parallelization. Training: Pretrain->SFT->RLHF."},
  {"source":"rag_design.md","content":"# RAG Design\nRAG combines retrieval with generation. Components: Document Pipeline, Search Engine, LLM Generator. 2026 standard: Hybrid Search + Rerank + Self-RAG."},
  {"source":"mcp_protocol.md","content":"# MCP Protocol\nMCP is AI Agent standard protocol. Linux Foundation (Dec 2025). 3 primitives: Tool, Resource, Prompt. 10,000+ servers, 97M monthly SDK downloads."},
]

pipeline.ingest_texts(KNOWLEDGE)
stats = pipeline.get_stats()
print(f"Pipeline: {stats["documents"]} docs -> {stats["total_chunks"]} chunks ({stats["total_chars"]} chars)")
print(f"Avg chunk: {stats["avg_chunk_size"]:.0f} chars")


In [ ]:
print("===== Chunk Quality Analysis =====\n")
for doc in pipeline.documents:
    sizes = [len(c) for c in doc.chunks]
    print(f"{doc.metadata["source"]}: {len(doc.chunks)} chunks, range [{min(sizes)},{max(sizes)}]")
    for i, c in enumerate(doc.chunks[:2]):
        print(f"  Chunk[{i}]: {c[:80]}...")


In [ ]:
from agent_project.hybrid_search import HybridSearcher
searcher = HybridSearcher()
searcher.index(pipeline.all_chunks)

results = searcher.search("What is RAG?", top_k=3)
print(f"Search test: {len(results)} results")
for r in results:
    print(f"  [{r["score"]:.4f}] {r["content"][:80]}...")
print("\nConnected: DocumentPipeline -> HybridSearcher -> Final Project!")
